# R26-DS-004 — model 2: XLM-RoBERTa-base (sequence classification)

**Same data as the TF–IDF baseline:** loads `text_primary` + `label` from the shared Parquet cache (no re-preprocess).

**Before you run:** On Google Drive under `Research Project Files/`, create a folder whose name matches **`MODEL_FOLDER_NAME`** in cell 1 (default: `xlm_roberta_base_model`).

**GPU:** Runtime → Change runtime type → **GPU** (T4 is enough to start). Training checkpoints go to **`/content/...`** for speed; the **final** model is copied to your model folder on Drive.

**Compare:** Cell 10 appends one row to **`model_runs_compare.csv`** next to your TF–IDF run.

**Load report (UNEXPECTED / MISSING):** Normal when loading `xlm-roberta-base` (masked LM) into a sequence-classification head — the `classifier.*` weights start random; `lm_head` / `pooler` from pretraining are dropped.

In [ ]:
# === CELL 1 — paths + training knobs (edit MODEL_FOLDER_NAME to match your Drive folder) ===
from pathlib import Path


def _ensure_colab_drive_mounted() -> None:
    try:
        from google.colab import drive
    except ImportError:
        return
    if Path("/content/drive/MyDrive").is_dir():
        return
    print("Connecting to Google Drive…")
    drive.mount("/content/drive")


def _drive_root() -> Path:
    for candidate in (Path("/content/drive/MyDrive"), Path("/content/drive/My drive")):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError("Mount Drive, then re-run this cell.")


_ensure_colab_drive_mounted()
DRIVE_ROOT = _drive_root()

RESEARCH_ROOT = DRIVE_ROOT / "Research Project Files"
CACHE_ROOT = RESEARCH_ROOT / "transaction_semantic_cache"
TRAIN_PQ = CACHE_ROOT / "train_preprocessed.parquet"
VAL_PQ = CACHE_ROOT / "val_preprocessed.parquet"
TEST_PQ = CACHE_ROOT / "test_preprocessed.parquet"
MANIFEST_JSON = CACHE_ROOT / "preprocess_manifest.json"

MODEL_FOLDER_NAME = "xlm_roberta_base_model"
MODEL_RUN_ROOT = RESEARCH_ROOT / MODEL_FOLDER_NAME
ARTIFACT_DIR = MODEL_RUN_ROOT / "artifacts"
EXPORT_DIR = MODEL_RUN_ROOT / "export"

METRICS_JSON = EXPORT_DIR / "metrics.json"
RUN_SUMMARY_CSV = EXPORT_DIR / "run_summary.csv"
LABEL_MAP_JSON = EXPORT_DIR / "label2id.json"
PER_CLASS_F1_CSV = EXPORT_DIR / "per_class_f1_test.csv"
MODEL_RUNS_MASTER_CSV = RESEARCH_ROOT / "model_runs_compare.csv"

HF_MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 256
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED = 42

HF_WORK_DIR = Path("/content/hf_xlmr_work")
FINAL_HF_DIR = ARTIFACT_DIR / "hf_model"

print("MODEL_RUN_ROOT =", MODEL_RUN_ROOT)
print("HF_MODEL_NAME  =", HF_MODEL_NAME)
print("HF_WORK_DIR    =", HF_WORK_DIR, "(fast local disk)")
print("FINAL_HF_DIR   =", FINAL_HF_DIR, "(on Drive)")

In [ ]:
# === CELL 2 — verify Parquet cache ===
missing = [p for p in (TRAIN_PQ, VAL_PQ, TEST_PQ, MANIFEST_JSON) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing preprocess outputs. Run preprocess first:\n" + "\n".join(str(p) for p in missing)
    )
print("OK — Parquet + manifest found.")

In [ ]:
# === CELL 3 — installs (once per runtime; GPU runtime recommended) ===
%pip install -q pandas pyarrow transformers datasets accelerate evaluate scikit-learn tqdm

In [ ]:
# === CELL 4 — imports ===
import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import classification_report, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(SEED)
print("OK — torch cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# === CELL 5 — load data + label mapping ===
train_df = pd.read_parquet(TRAIN_PQ)
val_df = pd.read_parquet(VAL_PQ)
test_df = pd.read_parquet(TEST_PQ)

for name, df in (("train", train_df), ("val", val_df), ("test", test_df)):
    if "text_primary" not in df.columns or "label" not in df.columns:
        raise ValueError(f"{name} missing text_primary or label")
    print(name, len(df))

all_labels = sorted(
    pd.concat([train_df["label"], val_df["label"], test_df["label"]]).astype(str).unique()
)
label2id = {lab: i for i, lab in enumerate(all_labels)}
id2label = {i: lab for lab, i in label2id.items()}
num_labels = len(all_labels)

train_df = train_df.assign(label_id=train_df["label"].astype(str).map(label2id))
val_df = val_df.assign(label_id=val_df["label"].astype(str).map(label2id))
test_df = test_df.assign(label_id=test_df["label"].astype(str).map(label2id))

if train_df["label_id"].isna().any():
    raise ValueError("Train has labels not in label2id")
for split_name, sdf in (("val", val_df), ("test", test_df)):
    if sdf["label_id"].isna().any():
        bad = sdf.loc[sdf["label_id"].isna(), "label"].unique()
        raise ValueError(f"{split_name} has unseen labels vs train union: {bad}")

print("num_labels =", num_labels)
print("OK — loaded")

In [ ]:
# === CELL 6 — tokenizer + Hugging Face datasets ===
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)

# HF Trainer + sequence classification expect column name "labels" (loss is skipped if missing).
ds_train = Dataset.from_pandas(
    train_df[["text_primary", "label_id"]].rename(columns={"text_primary": "text", "label_id": "labels"})
)
ds_val = Dataset.from_pandas(
    val_df[["text_primary", "label_id"]].rename(columns={"text_primary": "text", "label_id": "labels"})
)
ds_test = Dataset.from_pandas(
    test_df[["text_primary", "label_id"]].rename(columns={"text_primary": "text", "label_id": "labels"})
)


def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


ds_train = ds_train.map(tokenize_fn, batched=True, remove_columns=["text"])
ds_val = ds_val.map(tokenize_fn, batched=True, remove_columns=["text"])
ds_test = ds_test.map(tokenize_fn, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("OK — tokenized")

In [ ]:
# === CELL 7 — model + Trainer ===
import inspect
import math

model = AutoModelForSequenceClassification.from_pretrained(
    HF_MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    problem_type="single_label_classification",
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro = f1_score(labels, preds, average="macro", zero_division=0)
    weighted = f1_score(labels, preds, average="weighted", zero_division=0)
    return {"f1_macro": macro, "f1_weighted": weighted}


if HF_WORK_DIR.exists():
    shutil.rmtree(HF_WORK_DIR)
HF_WORK_DIR.mkdir(parents=True, exist_ok=True)

# Optimizer steps ≈ epochs * ceil(n / (batch * accum * devices)) — matches former warmup_ratio
_n_dev = max(1, torch.cuda.device_count())
_spe = math.ceil(len(ds_train) / (TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS * _n_dev))
_total_steps = max(1, _spe * NUM_EPOCHS)
warmup_steps = max(1, int(_total_steps * WARMUP_RATIO))

args = TrainingArguments(
    output_dir=str(HF_WORK_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

_trainer_kw = dict(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    _trainer_kw["processing_class"] = tokenizer  # Transformers v5+
else:
    _trainer_kw["tokenizer"] = tokenizer  # older Transformers
trainer = Trainer(**_trainer_kw)

print("OK — Trainer ready")

In [ ]:
# === CELL 8 — train (uses GPU if available) ===
train_result = trainer.train()
print("train_loss:", train_result.training_loss)
print("OK — training finished")

In [ ]:
# === CELL 9 — val + test metrics ===
val_metrics = trainer.evaluate(eval_dataset=ds_val)
print("val:", {k: round(v, 4) if isinstance(v, float) else v for k, v in val_metrics.items()})

test_pred = trainer.predict(ds_test)
logits = test_pred.predictions
y_true = test_pred.label_ids
y_hat = np.argmax(logits, axis=-1)

val_f1 = float(val_metrics.get("eval_f1_macro", 0.0))
test_f1 = float(f1_score(y_true, y_hat, average="macro", zero_division=0))
weighted_f1_test = float(f1_score(y_true, y_hat, average="weighted", zero_division=0))

y_test_str = test_df["label"].astype(str).values
y_hat_str = np.array([id2label[i] for i in y_hat])

print("macro-F1 test:", round(test_f1, 4))
print("weighted-F1 test:", round(weighted_f1_test, 4))
print("\n--- classification report (test) ---")
print(classification_report(y_test_str, y_hat_str, zero_division=0))

report = classification_report(
    y_test_str, y_hat_str, output_dict=True, zero_division=0
)
per_class_rows = []
for k, v in report.items():
    if isinstance(v, dict) and "f1-score" in v:
        per_class_rows.append({"label": k, "precision": v["precision"], "recall": v["recall"], "f1": v["f1-score"], "support": v["support"]})

print("OK — evaluated")

In [ ]:
# === CELL 10 — save to Drive + CSVs + master compare row ===
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_HF_DIR))
tokenizer.save_pretrained(str(FINAL_HF_DIR))

label_map_payload = {"label2id": label2id, "id2label": {str(k): v for k, v in id2label.items()}, "hf_model_name": HF_MODEL_NAME}
LABEL_MAP_JSON.write_text(json.dumps(label_map_payload, indent=2), encoding="utf-8")

metrics = {
    "model_id": "xlm_roberta_hf",
    "model_folder": MODEL_FOLDER_NAME,
    "hf_model_name": HF_MODEL_NAME,
    "macro_f1_val": val_f1,
    "macro_f1_test": test_f1,
    "weighted_f1_test": weighted_f1_test,
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
    "num_labels": num_labels,
    "model_path": str(FINAL_HF_DIR),
}
METRICS_JSON.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

pd.DataFrame(per_class_rows).to_csv(PER_CLASS_F1_CSV, index=False)

summary = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": "xlm_roberta_hf",
    "model_folder": MODEL_FOLDER_NAME,
    "hf_model_name": HF_MODEL_NAME,
    "n_vocab_features": "",  # align column with TF–IDF sheet; empty for transformers
    "n_label_classes": num_labels,
    "macro_f1_val": round(val_f1, 6),
    "macro_f1_test": round(test_f1, 6),
    "weighted_f1_test": round(weighted_f1_test, 6),
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_test": int(len(test_df)),
    "max_length": MAX_LENGTH,
    "num_epochs": NUM_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": warmup_steps,
    "warmup_ratio_equivalent": WARMUP_RATIO,
    "seed": SEED,
}
summary_df = pd.DataFrame([summary])
summary_df.to_csv(RUN_SUMMARY_CSV, index=False)

if MODEL_RUNS_MASTER_CSV.is_file():
    old = pd.read_csv(MODEL_RUNS_MASTER_CSV)
    master_df = pd.concat([old, summary_df], ignore_index=True, sort=False)
else:
    master_df = summary_df
master_df.to_csv(MODEL_RUNS_MASTER_CSV, index=False)

print("OK — saved model to", FINAL_HF_DIR)
print("metrics:", METRICS_JSON)
print("run_summary:", RUN_SUMMARY_CSV)
print("per_class_f1:", PER_CLASS_F1_CSV)
print("master compare rows:", len(master_df), "->", MODEL_RUNS_MASTER_CSV)